In [1]:
import pandas as pd
from pathlib import Path

# Load the data
belief_df = pd.read_csv('../data/raw/ml_belief_2024/data_release/belief_data.csv')
ratings_df = pd.read_csv('../data/raw/ml_belief_2024/data_release/user_rating_history.csv')
add_ratings_df = pd.read_csv('../data/raw/ml_belief_2024/data_release/ratings_for_additional_users.csv')
recommendations_df = pd.read_csv('../data/raw/ml_belief_2024/data_release/user_recommendation_history.csv')
movies_df = pd.read_csv('../data/raw/ml_belief_2024/data_release/movies.csv')

In [2]:
timestamp_cols = {
    "belief": belief_df["tstamp"],
    "ratings": ratings_df["tstamp"],
    "additional_ratings": add_ratings_df["tstamp"],
    "recommendations": recommendations_df["tstamp"],
}

for name, series in timestamp_cols.items():
    print(name)
    print("dtype:", series.dtype)
    print("First 5 values:")
    print(series.head().to_list())
    print("Min:", series.min())
    print("Max:", series.max())

belief
dtype: str
First 5 values:
['2023-05-01 18:59:04', '2023-10-08 13:52:36', '2023-12-21 19:02:56', '2023-06-22 23:31:47', '2023-12-31 19:33:33']
Min: 2023-03-13 19:00:14
Max: 2024-05-01 20:37:23
ratings
dtype: str
First 5 values:
['1998-06-18 16:31:37', '1998-06-18 16:31:37', '1998-06-18 16:31:37', '1997-11-07 13:41:17', '1997-11-07 13:27:51']
Min: 1997-09-17 17:53:58
Max: 2024-05-05 21:45:54
additional_ratings
dtype: str
First 5 values:
['2023-01-25 19:45:46', '2023-02-07 21:17:19', '2023-01-25 19:50:40', '2023-01-25 19:49:45', '2023-01-25 19:46:01']
Min: 1997-09-15 16:09:20
Max: 2024-05-05 21:44:02
recommendations
dtype: int64
First 5 values:
[1677651051, 1677651051, 1677651051, 1677651051, 1677651051]
Min: 1677651051
Max: 1714521553


## Data Cleaning

In [3]:
# Clean and standardize the belief data

belief_clean = belief_df.copy()

# Standardize naming
belief_clean = belief_clean.rename(
    columns={
        "userId": "user_id",
        "movieId": "movie_id",
        "isSeen": "is_seen",
        "watchDate": "watch_date",
        "userElicitRating": "user_elicit_rating",
        "userPredictRating": "user_predict_rating",
        "userCertainty": "user_certainty",
        "systemPredictRating": "system_predict_rating",
    }
)

# Parse timestamps
belief_clean["tstamp"] = pd.to_datetime(
    belief_clean["tstamp"], errors="coerce"
)

belief_clean["watch_date"] = pd.to_datetime(
    belief_clean["watch_date"], errors="coerce"
)

# Remove exact full-row duplicates
belief_rows_before_dedup = len(belief_clean)

belief_clean = belief_clean.drop_duplicates().copy()

belief_exact_duplicates_removed = (
    belief_rows_before_dedup - len(belief_clean)
)

# Remove inconsistent records
invalid_belief = (
    # isSeen = 1 but watchDate is missing
    (
        (belief_clean["is_seen"] == 1) &
        (belief_clean["watch_date"].isna())
    )

    |

    # isSeen = 0 but watchDate is populated
    (
        (belief_clean["is_seen"] == 0) &
        (belief_clean["watch_date"].notna())
    )

    |

    # isSeen = 0 but userElicitRating is populated
    (
        (belief_clean["is_seen"] == 0) &
        (belief_clean["user_elicit_rating"].notna()) &
        (belief_clean["user_elicit_rating"] != 0)
    )
)

belief_invalid_rows_removed = invalid_belief.sum()

belief_clean = belief_clean[~invalid_belief].copy()

# Summary
print("Original rows:", len(belief_df))
print("Exact duplicates removed:", belief_exact_duplicates_removed)
print("Invalid rows removed:", belief_invalid_rows_removed)
print("Final rows:", len(belief_clean))

Original rows: 3004084
Exact duplicates removed: 0
Invalid rows removed: 109
Final rows: 3003975


In [4]:
# Validate cleaned belief data

# 1. No exact full-row duplicates remain
assert not belief_clean.duplicated().any(), \
    "Exact duplicates remain in belief_clean."

# 2. No isSeen/watchDate inconsistencies remain
assert not (
    ((belief_clean["is_seen"] == 1) & belief_clean["watch_date"].isna()) |
    ((belief_clean["is_seen"] == 0) & belief_clean["watch_date"].notna())
).any(), \
    "isSeen/watchDate inconsistencies remain."

# 3. No unexpected elicited ratings for unseen movies remain
assert not (
    (belief_clean["is_seen"] == 0) &
    belief_clean["user_elicit_rating"].notna() &
    (belief_clean["user_elicit_rating"] != 0)
).any(), \
    "Unexpected elicited ratings remain for unseen movies."

# 4. No rows were unexpectedly removed
assert len(belief_df) - len(belief_clean) == 109, \
    "Unexpected number of belief records removed."

print("All belief_clean assertions passed.")

All belief_clean assertions passed.


In [5]:
# Clean and standardize the ratings data

ratings_clean = ratings_df.copy()

# Standardize naming
ratings_clean = ratings_clean.rename(
    columns={
        "userId": "user_id",
        "movieId": "movie_id",
    }
)

# Parse timestamp
ratings_clean["tstamp"] = pd.to_datetime(
    ratings_clean["tstamp"],
    errors="coerce"
)

# Remove only exact full-row duplicates
ratings_rows_before_dedup = len(ratings_clean)

ratings_clean = ratings_clean.drop_duplicates().copy()

ratings_exact_duplicates_removed = (
    ratings_rows_before_dedup - len(ratings_clean)
)

print("Original rows:", len(ratings_df))
print("Exact duplicates removed:", ratings_exact_duplicates_removed)
print("Final rows:", len(ratings_clean))

Original rows: 2046124
Exact duplicates removed: 41815
Final rows: 2004309


In [6]:
# Validate cleaned ratings data

# 1. No exact full-row duplicates remain
assert not ratings_clean.duplicated().any(), \
    "Exact duplicates remain in ratings_clean."

# 2. Only exact duplicates were removed
assert len(ratings_df) - len(ratings_clean) == ratings_exact_duplicates_removed, \
    "Unexpected number of rating records removed."

# 3. -1 ratings were not specifically filtered out
assert (
    (ratings_clean["rating"] == -1).sum()
    <=
    (ratings_df["rating"] == -1).sum()
), \
    "Unexpected change in -1 ratings."

print("All ratings_clean assertions passed.")

All ratings_clean assertions passed.


In [7]:
print(add_ratings_df.shape)
print(add_ratings_df.columns)
print(add_ratings_df.head())

(4185688, 4)
Index(['userId', 'movieId', 'rating', 'tstamp'], dtype='str')
   userId  movieId  rating               tstamp
0  393217        1     3.5  2023-01-25 19:45:46
1  393217        6     4.0  2023-02-07 21:17:19
2  393217       16     3.5  2023-01-25 19:50:40
3  393217       17     4.5  2023-01-25 19:49:45
4  393217       32     3.0  2023-01-25 19:46:01


In [8]:
print(
    "Users overlapping with ratings:",
    len(
        set(add_ratings_df["userId"])
        & set(ratings_clean["user_id"])
    )
)

print(
    "Users overlapping with beliefs:",
    len(
        set(add_ratings_df["userId"])
        & set(belief_clean["user_id"])
    )
)

Users overlapping with ratings: 0
Users overlapping with beliefs: 12490


In [9]:
# Clean and standardize the additional ratings data

add_ratings_clean = add_ratings_df.copy()

# Standardize naming
add_ratings_clean = add_ratings_clean.rename(
    columns={
        "userId": "user_id",
        "movieId": "movie_id",
    }
)

# Parse timestamp
add_ratings_clean["tstamp"] = pd.to_datetime(
    add_ratings_clean["tstamp"],
    errors="coerce"
)

# Remove only exact full-row duplicates
add_ratings_rows_before_dedup = len(add_ratings_clean)

add_ratings_clean = add_ratings_clean.drop_duplicates().copy()

add_ratings_exact_duplicates_removed = (
    add_ratings_rows_before_dedup - len(add_ratings_clean)
)

print("Original rows:", len(add_ratings_df))
print(
    "Exact duplicates removed:",
    add_ratings_exact_duplicates_removed
)
print("Final rows:", len(add_ratings_clean))

Original rows: 4185688
Exact duplicates removed: 0
Final rows: 4185688


In [10]:
# Validate cleaned additional ratings data

# No exact full-row duplicates remain
assert not add_ratings_clean.duplicated().any(), \
    "Exact duplicates remain in add_ratings_clean."

# Only exact duplicates were removed
assert len(add_ratings_df) - len(add_ratings_clean) == add_ratings_exact_duplicates_removed, \
    "Unexpected number of additional rating records removed."

print("All add_ratings_clean assertions passed.")

All add_ratings_clean assertions passed.


### Additional Ratings Data

`additional_ratings_df` is retained because it provides rating history for the same users represented in `belief_df`, while containing a distinct set of users from `ratings_df`. The table provides additional rating information that is not duplicated by the main `ratings_df` table.

In [11]:
# Clean and standardize the recommendations data

recommendations_clean = recommendations_df.copy()

# Standardize naming
recommendations_clean = recommendations_clean.rename(
    columns={
        "userId": "user_id",
        "movieId": "movie_id",
    }
)

# Parse timestamp
recommendations_clean["tstamp"] = pd.to_datetime(
    recommendations_clean["tstamp"],
    unit="s",
    errors="coerce"
)

# Remove only exact full-row duplicates
recommendations_rows_before_dedup = len(recommendations_clean)

recommendations_clean = recommendations_clean.drop_duplicates().copy()

recommendations_exact_duplicates_removed = (
    recommendations_rows_before_dedup - len(recommendations_clean)
)

print("Original rows:", len(recommendations_df))
print(
    "Exact duplicates removed:",
    recommendations_exact_duplicates_removed
)
print("Final rows:", len(recommendations_clean))

Original rows: 1285664
Exact duplicates removed: 312
Final rows: 1285352


In [12]:
# Validate cleaned recommendations data

# No exact full-row duplicates remain
assert not recommendations_clean.duplicated().any(), \
    "Exact duplicates remain in recommendations_clean."

# Only exact duplicates were removed
assert (
    len(recommendations_df) - len(recommendations_clean)
    == recommendations_exact_duplicates_removed
), \
    "Unexpected number of recommendation records removed."

print("All recommendations_clean assertions passed.")

All recommendations_clean assertions passed.


In [13]:
# Clean and standardize movie data

movies_clean = movies_df.copy()

# Standardize naming
movies_clean = movies_clean.rename(
    columns={
        "movieId": "movie_id"
    }
)

# Remove exact full-row duplicates
movies_rows_before_dedup = len(movies_clean)

movies_clean = movies_clean.drop_duplicates().copy()

movies_exact_duplicates_removed = (
    movies_rows_before_dedup - len(movies_clean)
)

print("Original rows:", len(movies_df))
print(
    "Exact duplicates removed:",
    movies_exact_duplicates_removed
)
print("Final rows:", len(movies_clean))

Original rows: 105071
Exact duplicates removed: 0
Final rows: 105071


In [14]:
# Handle missing genres by filling with "Unknown" and splitting the genres string into a list
movies_clean["genres"] = movies_clean["genres"].fillna("Unknown")
movies_clean["genres"] = movies_clean["genres"].apply(lambda x: x.split("|"))

# Group movies with no genres listed into a single "Unknown" category
movies_clean["genres"] = movies_clean["genres"].apply(
    lambda g: ["Unknown"] if g == ["(no genres listed)"] else g
)

In [15]:
# Validate cleaned movie data

# Exact duplicates were already removed before genre transformation
assert (
    len(movies_df) - len(movies_clean)
    == movies_exact_duplicates_removed
), "Unexpected number of movie records removed."

# Movie IDs should be unique
assert movies_clean["movie_id"].is_unique, \
    "Duplicate movie_id values remain in movies_clean."

print("All movies_clean assertions passed.")

All movies_clean assertions passed.


In [16]:
timestamp_cols_clean = {
    "belief": belief_clean["tstamp"],
    "ratings": ratings_clean["tstamp"],
    "additional_ratings": add_ratings_clean["tstamp"],
    "recommendations": recommendations_clean["tstamp"],
    "watch_date": belief_clean["watch_date"],
}

for name, series in timestamp_cols_clean.items():
    print(f"\n{name}")
    print("dtype:", series.dtype)
    print("min:", series.min())
    print("max:", series.max())
    print("missing:", series.isna().sum())


belief
dtype: datetime64[us]
min: 2023-03-13 19:00:14
max: 2024-05-01 20:37:23
missing: 0

ratings
dtype: datetime64[us]
min: 1997-09-17 17:53:58
max: 2024-05-05 21:45:54
missing: 0

additional_ratings
dtype: datetime64[us]
min: 1997-09-15 16:09:20
max: 2024-05-05 21:44:02
missing: 0

recommendations
dtype: datetime64[s]
min: 2023-03-01 06:10:51
max: 2024-04-30 23:59:13
missing: 0

watch_date
dtype: datetime64[us]
min: 1970-09-23 00:00:00
max: 2024-04-30 00:00:00
missing: 2995437


## Data Optimization

In [17]:
datasets = {
    "belief_clean": belief_clean,
    "ratings_clean": ratings_clean,
    "add_ratings_clean": add_ratings_clean,
    "recommendations_clean": recommendations_clean,
    "movies_clean": movies_clean
}

# Check data types of each dataset
for name, df in datasets.items():
    print(name)
    print(df.dtypes)

belief_clean
user_id                           int64
movie_id                          int64
is_seen                           int64
watch_date               datetime64[us]
user_elicit_rating              float64
user_predict_rating             float64
user_certainty                    int64
tstamp                   datetime64[us]
movie_idx                         int64
source                            int64
system_predict_rating           float64
dtype: object
ratings_clean
user_id              int64
movie_id             int64
rating             float64
tstamp      datetime64[us]
dtype: object
add_ratings_clean
user_id              int64
movie_id             int64
rating             float64
tstamp      datetime64[us]
dtype: object
recommendations_clean
user_id                    int64
tstamp             datetime64[s]
movie_id                   int64
predictedRating          float64
dtype: object
movies_clean
movie_id     int64
title          str
genres      object
dtype: object


In [18]:
# Check ranges of numeric columns in each dataset
for name, df in datasets.items():
    print(name)
    
    print(df.select_dtypes(include="number").describe().T[
        ["min", "max"]
    ])

belief_clean
                          min       max
user_id                1892.0  410460.0
movie_id                  1.0  299713.0
is_seen                  -1.0       1.0
user_elicit_rating       -1.0       5.0
user_predict_rating      -1.0       5.0
user_certainty           -1.0       5.0
movie_idx                 0.0      14.0
source                    1.0       3.0
system_predict_rating     0.0       5.0
ratings_clean
              min       max
user_id   42170.0  410572.0
movie_id      1.0  300341.0
rating       -1.0       5.0
add_ratings_clean
             min       max
user_id   1892.0  410453.0
movie_id     1.0  300349.0
rating      -1.0       5.0
recommendations_clean
                          min       max
user_id          42170.000000  410438.0
movie_id             1.000000  299065.0
predictedRating      0.444623       5.0
movies_clean
          min       max
movie_id  1.0  300373.0


In [19]:
# Check memory usage 
for name, df in datasets.items():
    memory_mb = df.memory_usage(deep=True).sum() / 1024**2
    print(f"{name}: {memory_mb:.2f} MB")

belief_clean: 275.02 MB
ratings_clean: 76.46 MB
add_ratings_clean: 127.74 MB
recommendations_clean: 49.03 MB
movies_clean: 18.32 MB


In [20]:
# Optimize memory usage

# Belief data
belief_clean["user_id"] = belief_clean["user_id"].astype("int32")
belief_clean["movie_id"] = belief_clean["movie_id"].astype("int32")
belief_clean["is_seen"] = belief_clean["is_seen"].astype("int8")
belief_clean["user_elicit_rating"] = belief_clean["user_elicit_rating"].astype("float32")
belief_clean["user_predict_rating"] = belief_clean["user_predict_rating"].astype("float32")
belief_clean["user_certainty"] = belief_clean["user_certainty"].astype("int8")
belief_clean["movie_idx"] = belief_clean["movie_idx"].astype("int8")
belief_clean["source"] = belief_clean["source"].astype("int8")
belief_clean["system_predict_rating"] = belief_clean["system_predict_rating"].astype("float32")

# Ratings data
ratings_clean["user_id"] = ratings_clean["user_id"].astype("int32")
ratings_clean["movie_id"] = ratings_clean["movie_id"].astype("int32")
ratings_clean["rating"] = ratings_clean["rating"].astype("float32")

# Additional ratings
add_ratings_clean["user_id"] = add_ratings_clean["user_id"].astype("int32")
add_ratings_clean["movie_id"] = add_ratings_clean["movie_id"].astype("int32")
add_ratings_clean["rating"] = add_ratings_clean["rating"].astype("float32")

# Recommendations
recommendations_clean["user_id"] = recommendations_clean["user_id"].astype("int32")
recommendations_clean["movie_id"] = recommendations_clean["movie_id"].astype("int32")
recommendations_clean["predictedRating"] = (
    recommendations_clean["predictedRating"].astype("float32")
)

# Movies
movies_clean["movie_id"] = movies_clean["movie_id"].astype("int32")

In [21]:
# Verify memory usage after optimization
datasets = {
    "belief_clean": belief_clean,
    "ratings_clean": ratings_clean,
    "add_ratings_clean": add_ratings_clean,
    "recommendations_clean": recommendations_clean,
    "movies_clean": movies_clean
}

for name, df in datasets.items():
    memory_mb = df.memory_usage(deep=True).sum() / 1024**2
    print(f"{name}: {memory_mb:.2f} MB")

belief_clean: 137.51 MB
ratings_clean: 53.52 MB
add_ratings_clean: 79.84 MB
recommendations_clean: 34.32 MB
movies_clean: 17.92 MB


In [22]:
# Create processed data directory
processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

# Save analysis-ready tables
belief_clean.to_parquet(
    processed_dir / "belief_clean.parquet",
    index=False
)

ratings_clean.to_parquet(
    processed_dir / "ratings_clean.parquet",
    index=False
)

add_ratings_clean.to_parquet(
    processed_dir / "add_ratings_clean.parquet",
    index=False
)

recommendations_clean.to_parquet(
    processed_dir / "recommendations_clean.parquet",
    index=False
)

movies_clean.to_parquet(
    processed_dir / "movies_clean.parquet",
    index=False
)

print("Analysis-ready datasets saved successfully.")

Analysis-ready datasets saved successfully.
